[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CeatursHarmginton/colab-gg/blob/main/ColabSteam.ipynb)

# 🎮 ColabGG Cloud Gaming Server Launcher (Kiểu VaultBox)

1. Chọn GPU cho Colab: **Runtime → Change runtime type → T4 GPU**.
2. Bấm chạy ô **Khởi chạy ColabGG Server Launcher** bên dưới.
3. Khi thẻ **ColabGG Cloud Gaming Server is Ready** hiện ra, mã `COLABGG_CONNECT` sẽ tự động được copy vào Clipboard — bạn chỉ cần bấm **🚀 Mở Web Client & Kết nối ngay** (hoặc mở Web Client và dán vào) để vào chơi!

In [ ]:
#@title 🚀 Khởi chạy ColabGG Server Launcher (Kiểu VaultBox — Auto Room ID & Token) { display-mode: "form" }
Relay_Server_URL = "https://colabgg-relay.lefantasusdelimei.workers.dev" #@param {type:"string"}
Custom_Room_ID = "" #@param {type:"string"}
Mount_Google_Drive = False #@param {type:"boolean"}
Restore_Backup_From_Drive = False #@param {type:"boolean"}
Backup_To_Drive_On_Stop = False #@param {type:"boolean"}
CF_TURN_KEY_ID = "" #@param {type:"string"}
CF_TURN_API_TOKEN = "" #@param {type:"string"}
CF_ZeroTrust_Tunnel_Token = "" #@param {type:"string"}

import base64, html, json, math, os, pathlib, secrets, signal, subprocess, sys, time
from IPython.display import HTML, Javascript, display

if Mount_Google_Drive:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted at /content/drive")

SCRIPT_CONTENT = "#!/usr/bin/env bash\n# ==============================================================================\n# ColabGG Cloud Gaming \u2014 Fixed & Relay-Enabled Edition\n# Supports:\n#   1) VaultBox-style Cloudflare Relay Room + WebRTC UDP (No trycloudflare!)\n#   2) Optional Cloudflare Zero Trust Named Tunnel (WARP + Native Moonlight App)\n#   3) Legacy Quick Tunnel fallback\n# ==============================================================================\n\nset -uo pipefail\n\nif [ ! -d \"/content\" ] && [ \"${FORCE_LINUX:-0}\" != \"1\" ]; then\n    echo \"======================================================================\"\n    echo \"[ERROR] Script nay duoc thiet ke de chay tren Google Colab (Linux GPU)!\"\n    echo \"        This script must be run inside a Google Colab GPU runtime.\"\n    echo \"        (Runtime -> Change runtime type -> Hardware accelerator: T4 GPU)\"\n    echo \"======================================================================\"\n    exit 1\nfi\n\nexport DEBIAN_FRONTEND=noninteractive\nexport PATH=\"$PATH:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/lib/games\"\n\nSTATE_FILE=\"/var/tmp/colab_gpu_setup.done\"\nLOG_FILE=\"/tmp/colab_gaming_setup.log\"\nPACKAGES_URL=\"https://github.com/kmille36/Colab-Cloud-Gaming/releases/download/1.0.0/t4packages.tar.gz\"\nPACKAGES_FALLBACK_URL=\"https://bit.ly/4dRt1Kn\"\nMOONWEB_ZIP_URL=\"https://github.com/kmille36/moonlight-web-remote/releases/download/0.0.2/colab-linux.zip\"\nMOONWEB_FALLBACK_URL=\"https://bit.ly/4fuW6fO\"\nCLOUDFLARED_URL=\"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\"\n\nWEB_USER=\"${WEB_USER:-admin}\"\nWEB_PASS=\"${WEB_PASS:-admin}\"\nSUNSHINE_USER=\"${SUNSHINE_USER:-admin}\"\nSUNSHINE_PASS=\"${SUNSHINE_PASS:-admin}\"\nRESTORE_BACKUP=\"${RESTORE_BACKUP:-ask}\"\nBACKUP_ON_EXIT=\"${BACKUP_ON_EXIT:-ask}\"\nAUTO_PAIR=\"${AUTO_PAIR:-true}\"\nKEEP_ALIVE_LOOP=\"${KEEP_ALIVE_LOOP:-true}\"\n\n# Relay & Cloudflare TURN / Zero Trust options\nCOLABGG_RELAY_URL=\"${COLABGG_RELAY_URL:-}\"\nCOLABGG_ROOM_ID=\"${COLABGG_ROOM_ID:-}\"\nCOLABGG_TOKEN=\"${COLABGG_TOKEN:-}\"\nCF_TURN_KEY_ID=\"${CF_TURN_KEY_ID:-}\"\nCF_TURN_API_TOKEN=\"${CF_TURN_API_TOKEN:-}\"\nCF_TUNNEL_TOKEN=\"${CF_TUNNEL_TOKEN:-}\"\n\ntouch \"$STATE_FILE\" \"$LOG_FILE\"\n\nlog_info() { printf \"\\033[1;32m[INFO]\\033[0m %s\\n\" \"$*\"; }\nlog_warn() { printf \"\\033[1;33m[WARN]\\033[0m %s\\n\" \"$*\"; }\nlog_err()  { printf \"\\033[1;31m[ERROR]\\033[0m %s\\n\" \"$*\"; }\n\nalready_done() {\n    grep -qxF \"$1\" \"$STATE_FILE\" 2>/dev/null\n}\n\nmark_done() {\n    if ! already_done \"$1\"; then\n        echo \"$1\" >> \"$STATE_FILE\"\n    fi\n}\n\nrun_step() {\n    local desc=\"$1\"\n    local cmd=\"$2\"\n    log_info \"$desc...\"\n    echo \"=== [$(date -Iseconds)] $desc ===\" >> \"$LOG_FILE\"\n    if bash -c \"$cmd\" >> \"$LOG_FILE\" 2>&1; then\n        return 0\n    else\n        local rc=$?\n        log_err \"Step failed (exit $rc): $desc\"\n        log_err \"Last 25 log lines ($LOG_FILE):\"\n        tail -n 25 \"$LOG_FILE\" || true\n        return $rc\n    fi\n}\n\noptional_input() {\n    local prompt=\"$1\"\n    local default=\"${2:-n}\"\n    local input=\"\"\n    if [ -t 0 ]; then\n        printf \"%s\" \"$prompt\" >&2\n        read -r input || input=\"$default\"\n    else\n        input=\"$default\"\n    fi\n    input=$(printf \"%s\" \"$input\" | tr '[:upper:]' '[:lower:]')\n    printf \"%s\" \"${input:-$default}\"\n}\n\nuser_exists() {\n    id \"$1\" >/dev/null 2>&1\n}\n\nbackup_user() {\n    echo \"\"\n    log_info \"Stopping user session processes...\"\n    pkill -KILL -u user >/dev/null 2>&1 || true\n\n    if ! mountpoint -q /content/drive 2>/dev/null && [ ! -d /content/drive/MyDrive ]; then\n        log_warn \"Google Drive not mounted (/content/drive/MyDrive). Skipping backup.\"\n        exit 0\n    fi\n\n    local ans=\"n\"\n    if [ \"$BACKUP_ON_EXIT\" = \"true\" ] || [ \"$BACKUP_ON_EXIT\" = \"y\" ]; then\n        ans=\"y\"\n    elif [ \"$BACKUP_ON_EXIT\" = \"false\" ] || [ \"$BACKUP_ON_EXIT\" = \"n\" ]; then\n        ans=\"n\"\n    else\n        ans=$(optional_input \"Do you want to back up /home/user to Google Drive? (y/n): \" \"n\")\n    fi\n\n    if [ \"$ans\" = \"y\" ]; then\n        if [ ! -d /home/user ]; then\n            log_err \"/home/user does not exist.\"\n            exit 1\n        fi\n        log_info \"Backing up /home/user -> /content/drive/MyDrive/backup.tar.gz ...\"\n        tar -cf - -C /home/user ./ | pigz -6 > /content/drive/MyDrive/backup.tar.gz\n        log_info \"Backup completed: /content/drive/MyDrive/backup.tar.gz\"\n    else\n        log_info \"Backup skipped.\"\n    fi\n    exit 0\n}\n\nsetup_linux_user() {\n    echo \"root:123456\" | sudo chpasswd || true\n    if ! user_exists user; then\n        sudo useradd -m -s /bin/bash user\n    fi\n    echo \"user:123456\" | sudo chpasswd || true\n    for grp in sudo video render audio input plugdev; do\n        if getent group \"$grp\" >/dev/null 2>&1; then\n            sudo usermod -aG \"$grp\" user || true\n        fi\n    done\n    printf 'user ALL=(ALL) NOPASSWD:ALL\\n' | sudo tee /etc/sudoers.d/colab-user >/dev/null\n    sudo chmod 0440 /etc/sudoers.d/colab-user\n\n    sudo mkdir -p /home/user/.config/sunshine /home/user/Desktop\n    printf '%s\\n' \"--no-sandbox\" \"--disable-dev-shm-usage\" | sudo tee /home/user/.config/chrome-flags.conf >/dev/null\n    printf '%s\\n' \"--no-sandbox\" \"--disable-dev-shm-usage\" | sudo tee /home/user/.config/electron-flags.conf >/dev/null\n    sudo chown -R user:user /home/user\n}\n\nrestore_backup() {\n    if [ ! -d /content/drive/MyDrive ] || [ ! -f /content/drive/MyDrive/backup.tar.gz ]; then\n        return 0\n    fi\n    setup_linux_user\n    log_info \"Restoring /content/drive/MyDrive/backup.tar.gz -> /home/user ...\"\n    pigz -dc /content/drive/MyDrive/backup.tar.gz | sudo tar -xf - -C /home/user\n    sudo chown -R user:user /home/user\n    log_info \"Backup restored successfully.\"\n}\n\ninstall_matching_nvidia_driver() {\n    if already_done \"nvidia_driver_installed\" && [ -f /usr/lib/x86_64-linux-gnu/xorg/modules/drivers/nvidia_drv.so ] && nvidia-smi >/dev/null 2>&1; then\n        log_info \"NVIDIA Xorg/userspace driver already installed, skipping.\"\n        return 0\n    fi\n\n    local host_ver=\"\"\n    if [ -r /proc/driver/nvidia/version ]; then\n        host_ver=$(sed -n 's/.*Kernel Module  *\\([0-9][0-9.]*\\).*/\\1/p' /proc/driver/nvidia/version | head -n 1)\n    fi\n    if [ -z \"$host_ver\" ]; then\n        host_ver=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | head -n 1 | tr -d '[:space:]')\n    fi\n    if [ -z \"$host_ver\" ]; then\n        log_err \"Could not detect host NVIDIA kernel driver version!\"\n        return 1\n    fi\n\n    log_info \"Detected Colab host NVIDIA kernel module version: $host_ver\"\n\n    if [ -f /usr/lib/x86_64-linux-gnu/xorg/modules/drivers/nvidia_drv.so ] && strings /usr/lib/x86_64-linux-gnu/xorg/modules/drivers/nvidia_drv.so 2>/dev/null | grep -qF \"$host_ver\"; then\n        log_info \"Existing nvidia_drv.so already matches $host_ver!\"\n        mark_done \"nvidia_driver_installed\"\n        return 0\n    fi\n\n    local installer=\"/bakdeb/NVIDIA-Linux-x86_64-${host_ver}.run\"\n    if [ ! -f \"$installer\" ]; then\n        log_warn \"Bundled NVIDIA installer does not match host kernel ($host_ver). Downloading NVIDIA-$host_ver.run...\"\n        installer=\"/tmp/NVIDIA-Linux-x86_64-${host_ver}.run\"\n        if ! wget -q --retry-connrefused --tries=3 -O \"$installer\" \"https://us.download.nvidia.com/XFree86/Linux-x86_64/${host_ver}/NVIDIA-Linux-x86_64-${host_ver}.run\" && \\\n           ! wget -q --retry-connrefused --tries=3 -O \"$installer\" \"https://download.nvidia.com/XFree86/Linux-x86_64/${host_ver}/NVIDIA-Linux-x86_64-${host_ver}.run\"; then\n            rm -f \"$installer\"\n            local major=\"${host_ver%%.*}\"\n            run_step \"Install xserver-xorg-video-nvidia-${major}\" \"sudo apt-get update -y && sudo apt-get install -y xserver-xorg-video-nvidia-${major} libnvidia-gl-${major} libnvidia-gl-${major}:i386 libnvidia-encode-${major}\" || return 1\n            mark_done \"nvidia_driver_installed\"\n            return 0\n        fi\n    fi\n\n    chmod +x \"$installer\"\n    run_step \"Install NVIDIA userspace & Xorg driver ($host_ver)\" \\\n        \"sudo '$installer' --silent --no-kernel-module --install-compat32-libs --no-nouveau-check --no-nvidia-modprobe --no-rpms --no-backup\" || return 1\n\n    mark_done \"nvidia_driver_installed\"\n}\n\nsteam_check_loop() {\n    log_info \"Entering keep-alive & watchdog loop...\"\n    while true; do\n        if ! pgrep -u user -x steam >/dev/null 2>&1 && ! pgrep -u user -f \"ubuntu12_32/steam\" >/dev/null 2>&1; then\n            su - user -c 'DISPLAY=:1 nohup steam -console >/tmp/steam.log 2>&1 &' || true\n        fi\n        if ! pgrep -u user -x sunshine >/dev/null 2>&1; then\n            su - user -c 'DISPLAY=:1 nohup sunshine >/tmp/sunshine.log 2>&1 &' || true\n        fi\n        if [ -n \"$COLABGG_RELAY_URL\" ] && [ -n \"$COLABGG_ROOM_ID\" ] && ! pgrep -f \"colab_relay_agent.py\" >/dev/null 2>&1; then\n            log_warn \"Relay agent stopped. Restarting colab_relay_agent.py...\"\n            COLABGG_RELAY_URL=\"$COLABGG_RELAY_URL\" COLABGG_ROOM_ID=\"$COLABGG_ROOM_ID\" COLABGG_TOKEN=\"$COLABGG_TOKEN\" \\\n            CF_TURN_KEY_ID=\"$CF_TURN_KEY_ID\" CF_TURN_API_TOKEN=\"$CF_TURN_API_TOKEN\" WEB_USER=\"$WEB_USER\" WEB_PASS=\"$WEB_PASS\" \\\n            nohup python3 /content/colab_relay_agent.py >/tmp/colabgg_relay.log 2>&1 &\n        fi\n        sleep 25\n    done\n}\n\nmain() {\n    echo \"======================================================================\"\n    echo \"   ColabGG Cloud Gaming \u2014 Sunshine + WebRTC UDP + Cloudflare Relay\"\n    echo \"======================================================================\"\n\n    if ! nvidia-smi >/dev/null 2>&1; then\n        log_err \"No NVIDIA GPU detected! In Google Colab: Runtime -> Change runtime type -> T4 GPU.\"\n        exit 1\n    fi\n\n    # 1. Base CLI utilities\n    local missing_pkgs=()\n    for bin_pkg in \"wget:wget\" \"curl:curl\" \"pigz:pigz\" \"unzip:unzip\" \"jq:jq\" \"lspci:pciutils\" \"fuser:psmisc\"; do\n        local bin=\"${bin_pkg%%:*}\"\n        local pkg=\"${bin_pkg##*:}\"\n        if ! command -v \"$bin\" >/dev/null 2>&1; then\n            missing_pkgs+=(\"$pkg\")\n        fi\n    done\n    if [ \"${#missing_pkgs[@]}\" -gt 0 ]; then\n        run_step \"Install base prerequisite utilities (${missing_pkgs[*]})\" \\\n            \"sudo apt-get update -y && sudo apt-get install -y ${missing_pkgs[*]}\" || exit 1\n    fi\n\n    # 2. Safe /etc/profile.d/custom_path.sh\n    printf '#!/bin/sh\\nexport PATH=\"$PATH:/usr/games:/usr/lib/games\"\\n' | sudo tee /etc/profile.d/custom_path.sh >/dev/null\n    sudo chmod 0755 /etc/profile.d/custom_path.sh\n\n    # 3. Extract /bakdeb bundle once\n    if [ ! -d /bakdeb ] || ! already_done \"tar_extracted\"; then\n        run_step \"Download & extract gaming package bundle (~855 MB)\" \\\n            \"(wget -qO- --retry-connrefused --tries=3 --max-redirect=10 '$PACKAGES_URL' || wget -qO- --no-check-certificate --max-redirect=10 '$PACKAGES_FALLBACK_URL') | pigz -dc | sudo tar -xf - -C /\" || exit 1\n        mark_done \"tar_extracted\"\n    else\n        log_info \"Package bundle (/bakdeb) already extracted, skipping download.\"\n    fi\n\n    # 4. User setup & optional Google Drive backup restore\n    setup_linux_user\n    if [ -f /content/drive/MyDrive/backup.tar.gz ]; then\n        local do_restore=\"n\"\n        if [ \"$RESTORE_BACKUP\" = \"true\" ] || [ \"$RESTORE_BACKUP\" = \"y\" ]; then\n            do_restore=\"y\"\n        elif [ \"$RESTORE_BACKUP\" = \"false\" ] || [ \"$RESTORE_BACKUP\" = \"n\" ]; then\n            do_restore=\"n\"\n        else\n            do_restore=$(optional_input \"Found backup.tar.gz on Google Drive. Restore backup? (y/n): \" \"n\")\n        fi\n        if [ \"$do_restore\" = \"y\" ]; then\n            restore_backup\n        fi\n    fi\n\n    # 5. Install desktop/gaming .deb packages\n    if ! already_done \"packages_installed\"; then\n        run_step \"Enable i386 multiarch & install desktop/gaming packages\" \\\n            \"sudo dpkg --add-architecture i386 && (sudo DEBIAN_FRONTEND=noninteractive dpkg -i /bakdeb/*.deb || true) && sudo DEBIAN_FRONTEND=noninteractive apt-get install --fix-broken -y -o Dir::Cache::archives='/bakdeb'\" || \\\n        run_step \"Update apt cache & repair missing dependencies\" \\\n            \"sudo apt-get update -y && sudo DEBIAN_FRONTEND=noninteractive apt-get install --fix-broken -y\" || exit 1\n        mark_done \"packages_installed\"\n    else\n        log_info \"Desktop & gaming packages already installed, skipping.\"\n    fi\n\n    # 6. Install matching NVIDIA driver\n    install_matching_nvidia_driver || exit 1\n\n    # 7. Start virtual 1080p60 Xorg display (:1)\n    run_step \"Configure virtual 1920x1080 NVIDIA Xorg screen\" \\\n        \"sudo nvidia-xconfig -a --allow-empty-initial-configuration --virtual=1920x1080 --preserve-busid\" || true\n\n    if ! pgrep -f \"Xorg :1\" >/dev/null 2>&1; then\n        sudo rm -f /tmp/.X1-lock /tmp/.X11-unix/X1 || true\n        log_info \"Starting Xorg server on DISPLAY=:1 ...\"\n        sudo nohup Xorg :1 -seat seat-1 -novtswitch -nolisten tcp -noreset >/tmp/xorg.log 2>&1 &\n        for _ in $(seq 1 15); do\n            if DISPLAY=:1 xrandr >/dev/null 2>&1; then\n                break\n            fi\n            sleep 1\n        done\n    fi\n\n    if ! DISPLAY=:1 xrandr >/dev/null 2>&1; then\n        log_err \"Xorg :1 failed to start! Check /tmp/xorg.log:\"\n        tail -n 30 /tmp/xorg.log || true\n        exit 1\n    fi\n\n    DISPLAY=:1 xhost +local: >/dev/null 2>&1 || true\n    local xout=\"\"\n    xout=$(DISPLAY=:1 xrandr 2>/dev/null | awk '/ connected/{print $1; exit}')\n    if [ -n \"$xout\" ]; then\n        DISPLAY=:1 xrandr --output \"$xout\" --mode 1920x1080 --rate 60 >/dev/null 2>&1 || \\\n        DISPLAY=:1 xrandr --output \"$xout\" --mode 1920x1080 >/dev/null 2>&1 || true\n    fi\n\n    # 8. Start PulseAudio (virtual sink), Openbox, Sunshine, Steam, Heroic, Chrome\n    [ -f /bakdeb/nvidia-wallpaper.jpg ] && sudo cp -f /bakdeb/nvidia-wallpaper.jpg /home/user/nvidia-wallpaper.jpg && sudo chown user:user /home/user/nvidia-wallpaper.jpg\n\n    log_info \"Starting PulseAudio, Openbox, Sunshine, Steam & desktop tools...\"\n    su - user -c 'pulseaudio --check >/dev/null 2>&1 || pulseaudio --start --exit-idle-time=-1 >/dev/null 2>&1 || true'\n    su - user -c 'pactl list short sinks 2>/dev/null | grep -q sunshine_sink || pactl load-module module-null-sink sink_name=sunshine_sink sink_properties=device.description=\"Sunshine_Virtual_Audio\" >/dev/null 2>&1 || true'\n    su - user -c 'pactl set-default-sink sunshine_sink >/dev/null 2>&1 || true'\n\n    pgrep -u user -x openbox >/dev/null 2>&1 || su - user -c 'DISPLAY=:1 nohup openbox >/tmp/openbox.log 2>&1 &'\n    sleep 1\n    [ -f /home/user/nvidia-wallpaper.jpg ] && su - user -c 'DISPLAY=:1 feh --bg-scale ~/nvidia-wallpaper.jpg >/dev/null 2>&1 || true'\n\n    pkill -u user -x sunshine >/dev/null 2>&1 || true\n    sleep 1\n    su - user -c \"sunshine --creds '$SUNSHINE_USER' '$SUNSHINE_PASS'\" >/dev/null 2>&1 || true\n    su - user -c 'DISPLAY=:1 nohup sunshine >/tmp/sunshine.log 2>&1 &'\n\n    pgrep -u user -x steam >/dev/null 2>&1 || su - user -c 'DISPLAY=:1 nohup steam -console >/tmp/steam.log 2>&1 &'\n    pgrep -u user -x xvkbd >/dev/null 2>&1 || su - user -c 'DISPLAY=:1 nohup xvkbd >/dev/null 2>&1 &'\n    pgrep -u user -x thunar >/dev/null 2>&1 || su - user -c 'DISPLAY=:1 nohup thunar >/dev/null 2>&1 &'\n    if command -v heroic >/dev/null 2>&1 && ! pgrep -u user -f heroic >/dev/null 2>&1; then\n        su - user -c 'DISPLAY=:1 nohup heroic --no-sandbox --disable-dev-shm-usage >/tmp/heroic.log 2>&1 &' || true\n    fi\n    if command -v google-chrome >/dev/null 2>&1 && ! pgrep -u user -f google-chrome >/dev/null 2>&1; then\n        su - user -c 'DISPLAY=:1 nohup google-chrome --no-sandbox --disable-dev-shm-usage --no-first-run https://store.steampowered.com >/tmp/chrome.log 2>&1 &' || true\n    fi\n\n    # 9. Install, patch, and launch Moonlight Web Remote (:8081)\n    if [ ! -x /opt/moonlight-web/package/web-server ]; then\n        run_step \"Download & unpack Moonlight Web Remote\" \\\n            \"sudo rm -rf /opt/moonlight-web && (wget -q --retry-connrefused --tries=3 -O /tmp/colab-linux.zip -L '$MOONWEB_ZIP_URL' || wget --no-check-certificate -q -O /tmp/colab-linux.zip -L '$MOONWEB_FALLBACK_URL') && sudo unzip -qo /tmp/colab-linux.zip -d /opt/moonlight-web && rm -f /tmp/colab-linux.zip && sudo chmod +x /opt/moonlight-web/package/web-server /opt/moonlight-web/package/streamer\" || exit 1\n    fi\n    sudo python3 -c '\nimport pathlib\np = pathlib.Path(\"/opt/moonlight-web/package/static/api.js\")\nif p.exists():\n    s = p.read_text(encoding=\"utf-8\")\n    old = \"request.signal = AbortSignal.timeout(timeout);\"\n    new = \"if (!(init && init.noTimeout)) { request.signal = AbortSignal.timeout(timeout); }\"\n    if old in s and new not in s:\n        p.write_text(s.replace(old, new, 1), encoding=\"utf-8\")\nd = pathlib.Path(\"/opt/moonlight-web/package/static/default_settings.js\")\nif d.exists():\n    ds = d.read_text(encoding=\"utf-8\")\n    ds = ds.replace(\"\\\"dataTransport\\\": \\\"websocket\\\"\", \"\\\"dataTransport\\\": \\\"auto\\\"\", 1)\n    d.write_text(ds, encoding=\"utf-8\")\n'\n\n    pkill -f \"/opt/moonlight-web/package/web-server\" >/dev/null 2>&1 || true\n    sleep 1\n    log_info \"Starting local Moonlight Web Server on 0.0.0.0:8081 ...\"\n    (cd /opt/moonlight-web/package && nohup ./web-server --bind-address 0.0.0.0:8081 >/tmp/webserver.log 2>&1 &)\n\n    # 10. Auto-pair Moonlight Web (127.0.0.1) with Sunshine (:47990)\n    if [ \"$AUTO_PAIR\" = \"true\" ]; then\n        log_info \"Auto-pairing local Moonlight Web with Sunshine...\"\n        WEB_USER=\"$WEB_USER\" WEB_PASS=\"$WEB_PASS\" SUNSHINE_USER=\"$SUNSHINE_USER\" SUNSHINE_PASS=\"$SUNSHINE_PASS\" python3 - --auto <<'PYAUTOPAIR' || log_warn \"Auto-pair warning; can re-run moon-pair.sh\"\nimport base64\nimport json\nimport os\nimport ssl\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\n\nWEB_BASE = os.environ.get(\"MOONWEB_URL\", \"http://127.0.0.1:8081/api\")\nSUNSHINE_BASE = os.environ.get(\"SUNSHINE_URL\", \"https://127.0.0.1:47990/api\")\nWEB_USER = os.environ.get(\"WEB_USER\", \"admin\")\nWEB_PASS = os.environ.get(\"WEB_PASS\", \"admin\")\nSUN_USER = os.environ.get(\"SUNSHINE_USER\", \"admin\")\nSUN_PASS = os.environ.get(\"SUNSHINE_PASS\", \"admin\")\n\nssl_ctx = ssl.create_default_context()\nssl_ctx.check_hostname = False\nssl_ctx.verify_mode = ssl.CERT_NONE\n\n\ndef sunshine_req(path, payload):\n    auth = base64.b64encode(f\"{SUN_USER}:{SUN_PASS}\".encode()).decode()\n    data = json.dumps(payload).encode(\"utf-8\")\n    req = urllib.request.Request(\n        f\"{SUNSHINE_BASE}{path}\",\n        data=data,\n        method=\"POST\",\n        headers={\n            \"Content-Type\": \"application/json\",\n            \"Authorization\": f\"Basic {auth}\",\n        },\n    )\n    with urllib.request.urlopen(req, context=ssl_ctx, timeout=15) as resp:\n        return resp.status, resp.read().decode(\"utf-8\", errors=\"replace\")\n\n\ndef ensure_sunshine_ready():\n    for _ in range(25):\n        try:\n            req = urllib.request.Request(SUNSHINE_BASE.replace(\"/api\", \"/\"), method=\"GET\")\n            urllib.request.urlopen(req, context=ssl_ctx, timeout=3)\n            break\n        except urllib.error.HTTPError:\n            break\n        except Exception:\n            time.sleep(1)\n    try:\n        sunshine_req(\n            \"/password\",\n            {\n                \"currentUsername\": SUN_USER,\n                \"currentPassword\": SUN_PASS,\n                \"newUsername\": SUN_USER,\n                \"newPassword\": SUN_PASS,\n                \"confirmNewPassword\": SUN_PASS,\n            },\n        )\n    except Exception:\n        pass\n\n\ndef web_req(path, method=\"GET\", payload=None, cookie=\"\", timeout=15):\n    headers = {}\n    if cookie:\n        headers[\"Cookie\"] = cookie\n    data = None\n    if payload is not None:\n        headers[\"Content-Type\"] = \"application/json\"\n        data = json.dumps(payload).encode(\"utf-8\")\n    req = urllib.request.Request(f\"{WEB_BASE}{path}\", data=data, method=method, headers={**headers})\n    return urllib.request.urlopen(req, timeout=timeout)\n\n\ndef login_moonweb():\n    for _ in range(20):\n        try:\n            resp = web_req(\"/login\", method=\"POST\", payload={\"name\": WEB_USER, \"password\": WEB_PASS})\n            raw_cookies = resp.headers.get_all(\"Set-Cookie\") or []\n            cookie = \"; \".join(c.split(\";\", 1)[0] for c in raw_cookies if \"=\" in c)\n            return cookie\n        except urllib.error.HTTPError as e:\n            if e.code in (401, 403):\n                raise RuntimeError(f\"Moonlight Web login rejected for user '{WEB_USER}' (HTTP {e.code}).\")\n            time.sleep(1)\n        except Exception:\n            time.sleep(1)\n    raise RuntimeError(\"Moonlight Web server (:8081) did not become ready in time.\")\n\n\ndef main():\n    mode = sys.argv[1] if len(sys.argv) > 1 else \"--auto\"\n    ensure_sunshine_ready()\n\n    if mode != \"--auto\":\n        pin = mode.strip()\n        status, body = sunshine_req(\"/pin\", {\"pin\": pin, \"name\": \"moonlight\"})\n        print(f\"Sunshine PIN response ({status}): {body}\")\n        return 0\n\n    cookie = login_moonweb()\n\n    resp = web_req(\"/hosts\", method=\"GET\", cookie=cookie, timeout=15)\n    first_line = resp.readline().decode(\"utf-8\").strip()\n    resp.close()\n    hosts_obj = json.loads(first_line) if first_line else {\"hosts\": []}\n    hosts = hosts_obj.get(\"hosts\", [])\n\n    target_host = None\n    for h in hosts:\n        if h.get(\"address\") in (\"127.0.0.1\", \"localhost\") or len(hosts) == 1:\n            target_host = h\n            break\n\n    if target_host is None:\n        resp = web_req(\n            \"/host\",\n            method=\"POST\",\n            payload={\"address\": \"127.0.0.1\", \"http_port\": None},\n            cookie=cookie,\n            timeout=20,\n        )\n        target_host = json.loads(resp.read().decode(\"utf-8\"))[\"host\"]\n\n    host_id = target_host[\"host_id\"]\n\n    try:\n        web_req(\n            \"/host\",\n            method=\"PATCH\",\n            payload={\"host_id\": host_id, \"change_owner\": True, \"owner\": None},\n            cookie=cookie,\n            timeout=10,\n        )\n    except Exception:\n        pass\n\n    if target_host.get(\"paired\") == \"Paired\":\n        print(f\"[Auto-Pair] Host #{host_id} (127.0.0.1) is already Paired!\")\n        return 0\n\n    pair_resp = web_req(\n        \"/pair\",\n        method=\"POST\",\n        payload={\"host_id\": host_id},\n        cookie=cookie,\n        timeout=60,\n    )\n    stage1_raw = pair_resp.readline().decode(\"utf-8\").strip()\n    if not stage1_raw:\n        raise RuntimeError(\"Empty stage 1 response from /api/pair\")\n    stage1 = json.loads(stage1_raw)\n    if isinstance(stage1, str) or \"Pin\" not in stage1:\n        raise RuntimeError(f\"Failed to initiate pair (stage 1): {stage1}\")\n\n    pin = stage1[\"Pin\"]\n    print(f\"[Auto-Pair] Generated Moonlight PIN: {pin} -> Submitting to Sunshine...\")\n    sunshine_req(\"/pin\", {\"pin\": pin, \"name\": \"moonlight-web\"})\n\n    stage2_raw = pair_resp.readline().decode(\"utf-8\").strip()\n    pair_resp.close()\n    if not stage2_raw:\n        raise RuntimeError(\"Empty stage 2 response from /api/pair\")\n    stage2 = json.loads(stage2_raw)\n    if isinstance(stage2, str) or \"Paired\" not in stage2:\n        raise RuntimeError(f\"Failed to complete pair (stage 2): {stage2}\")\n\n    print(f\"[Auto-Pair] Successfully paired Moonlight Web host #{host_id} with Sunshine!\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\nPYAUTOPAIR\n    fi\n\n    # 11. Optional Cloudflare Zero Trust Named Tunnel (WARP + Native Moonlight App)\n    if [ -n \"$CF_TUNNEL_TOKEN\" ]; then\n        if [ ! -x /usr/local/bin/cloudflared ]; then\n            run_step \"Install cloudflared binary\" \\\n                \"sudo wget -q --retry-connrefused --tries=3 '$CLOUDFLARED_URL' -O /usr/local/bin/cloudflared && sudo chmod +x /usr/local/bin/cloudflared\" || true\n        fi\n        pkill -f \"cloudflared tunnel.*--token\" >/dev/null 2>&1 || true\n        log_info \"Starting Cloudflare Zero Trust Private Tunnel (QUIC)...\"\n        nohup /usr/local/bin/cloudflared tunnel --no-autoupdate --protocol quic run --token \"$CF_TUNNEL_TOKEN\" >/tmp/cf_zerotrust.log 2>&1 &\n        log_info \"Zero Trust Active! Colab Internal IP for Native Moonlight App: $(hostname -I | awk '{print $1}')\"\n    fi\n\n    # 12. Primary Connection Mode: VaultBox-style Cloudflare Relay Room OR Legacy Quick Tunnel\n    if [ -n \"$COLABGG_RELAY_URL\" ] && [ -n \"$COLABGG_ROOM_ID\" ] && [ -n \"$COLABGG_TOKEN\" ]; then\n        run_step \"Install Python websockets client for ColabGG Relay\" \\\n            \"python3 -m pip install -q websockets\" || exit 1\n\n        cat <<'PYRELAYEOF' > /content/colab_relay_agent.py\n#!/usr/bin/env python3\n\"\"\"\nColabGG Relay Agent \u2014 Bridges local Moonlight-Web (:8081) & Sunshine (:47990)\non Google Colab to the dedicated `colabgg-relay` Cloudflare Worker + Durable Object,\nplus configures Cloudflare TURN UDP (`webrtc.ice_servers`) automatically.\n\"\"\"\nfrom __future__ import annotations\n\nimport asyncio\nimport base64\nimport inspect\nimport json\nimport os\nimport pathlib\nimport subprocess\nimport sys\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\nRELAY_URL = os.environ.get(\"COLABGG_RELAY_URL\", \"\").strip().rstrip(\"/\")\nROOM_ID = os.environ.get(\"COLABGG_ROOM_ID\", \"\").strip()\nTOKEN = os.environ.get(\"COLABGG_TOKEN\", \"\").strip()\nLOCAL_ORIGIN = os.environ.get(\"MOONWEB_LOCAL_ORIGIN\", \"http://127.0.0.1:8081\").strip().rstrip(\"/\")\nWEB_USER = os.environ.get(\"WEB_USER\", \"admin\").strip() or \"admin\"\nWEB_PASS = os.environ.get(\"WEB_PASS\", \"admin\").strip() or \"admin\"\nCF_TURN_KEY_ID = os.environ.get(\"CF_TURN_KEY_ID\", \"\").strip()\nCF_TURN_API_TOKEN = os.environ.get(\"CF_TURN_API_TOKEN\", \"\").strip()\nMOONWEB_DIR = pathlib.Path(os.environ.get(\"MOONWEB_DIR\", \"/opt/moonlight-web/package\"))\n\n\ndef build_ws_url(role: str, extra_query: dict[str, str] | None = None) -> str:\n    raw = RELAY_URL\n    if \"://\" not in raw:\n        raw = \"https://\" + raw\n    parsed = urllib.parse.urlsplit(raw)\n    scheme = \"wss\" if parsed.scheme == \"https\" else \"ws\"\n    params = {\"room\": ROOM_ID, \"role\": role, \"token\": TOKEN}\n    if extra_query:\n        params.update(extra_query)\n    return urllib.parse.urlunsplit(\n        (scheme, parsed.netloc, \"/api/colabgg/ws\", urllib.parse.urlencode(params), \"\")\n    )\n\n\ndef local_ws_origin() -> str:\n    parsed = urllib.parse.urlsplit(LOCAL_ORIGIN)\n    scheme = \"wss\" if parsed.scheme == \"https\" else \"ws\"\n    return urllib.parse.urlunsplit((scheme, parsed.netloc, \"\", \"\", \"\"))\n\n\nclass LocalMoonWebClient:\n    def __init__(self) -> None:\n        self.cookie: str = \"\"\n        self.turn_applied: bool = False\n\n    def login(self) -> str:\n        payload = json.dumps({\"name\": WEB_USER, \"password\": WEB_PASS}).encode(\"utf-8\")\n        for _ in range(15):\n            try:\n                req = urllib.request.Request(\n                    f\"{LOCAL_ORIGIN}/api/login\",\n                    data=payload,\n                    method=\"POST\",\n                    headers={\"Content-Type\": \"application/json\"},\n                )\n                with urllib.request.urlopen(req, timeout=10) as resp:\n                    cookies = resp.headers.get_all(\"Set-Cookie\") or []\n                    self.cookie = \"; \".join(c.split(\";\", 1)[0] for c in cookies if \"=\" in c)\n                    return self.cookie\n            except Exception:\n                time.sleep(1)\n        return self.cookie\n\n    def ensure_cookie(self) -> str:\n        if not self.cookie:\n            return self.login()\n        return self.cookie\n\n    def request(self, method: str, path_and_query: str, headers: dict[str, str], body: bytes) -> tuple[int, dict[str, str], bytes]:\n        for attempt in range(2):\n            cookie = self.ensure_cookie()\n            req_headers = dict(headers or {})\n            if cookie:\n                req_headers[\"Cookie\"] = cookie\n            data = body if method not in (\"GET\", \"HEAD\") and body else None\n            req = urllib.request.Request(\n                f\"{LOCAL_ORIGIN}{path_and_query}\",\n                data=data,\n                method=method,\n                headers=req_headers,\n            )\n            try:\n                with urllib.request.urlopen(req, timeout=18) as resp:\n                    out_body = resp.read()\n                    out_headers = {k: v for k, v in resp.headers.items()}\n                    return resp.status, out_headers, out_body\n            except urllib.error.HTTPError as e:\n                if e.code == 401 and attempt == 0:\n                    self.cookie = \"\"\n                    continue\n                err_body = e.read() if e.fp else b\"\"\n                err_headers = {k: v for k, v in (e.headers.items() if e.headers else [])}\n                return e.code, err_headers, err_body\n            except Exception as exc:\n                if attempt == 0:\n                    time.sleep(0.4)\n                    continue\n                msg = json.dumps({\"error\": \"local_moonweb_error\", \"message\": str(exc)}).encode(\"utf-8\")\n                return 502, {\"content-type\": \"application/json\"}, msg\n        return 502, {\"content-type\": \"application/json\"}, b'{\"error\":\"unreachable\"}'\n\n    def get_apps(self) -> list[dict]:\n        status, _, body = self.request(\"GET\", \"/api/apps?host_id=0\", {}, b\"\")\n        if status == 200 and body:\n            try:\n                data = json.loads(body.decode(\"utf-8\"))\n                return data.get(\"apps\", []) if isinstance(data, dict) else []\n            except Exception:\n                pass\n        return []\n\n\ndef query_gpu_telemetry() -> tuple[str, str, str]:\n    gpu_name, driver_ver, priv_ip = \"\", \"\", \"\"\n    try:\n        out = subprocess.check_output(\n            [\"nvidia-smi\", \"--query-gpu=name,driver_version\", \"--format=csv,noheader\"],\n            text=True,\n            timeout=4,\n        ).strip()\n        if out:\n            parts = [p.strip() for p in out.splitlines()[0].split(\",\")]\n            gpu_name = parts[0] if len(parts) > 0 else \"\"\n            driver_ver = parts[1] if len(parts) > 1 else \"\"\n    except Exception:\n        pass\n    try:\n        ip_out = subprocess.check_output([\"hostname\", \"-I\"], text=True, timeout=3).strip().split()\n        if ip_out:\n            priv_ip = ip_out[0]\n    except Exception:\n        pass\n    return gpu_name, driver_ver, priv_ip\n\n\ndef fetch_direct_cloudflare_turn() -> list[dict]:\n    if not (CF_TURN_KEY_ID and CF_TURN_API_TOKEN):\n        return []\n    url = f\"https://rtc.live.cloudflare.com/v1/turn/keys/{urllib.parse.quote(CF_TURN_KEY_ID)}/credentials/generate\"\n    req = urllib.request.Request(\n        url,\n        data=b'{\"ttl\":86400}',\n        method=\"POST\",\n        headers={\n            \"Authorization\": f\"Bearer {CF_TURN_API_TOKEN}\",\n            \"Content-Type\": \"application/json\",\n        },\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=10) as resp:\n            data = json.loads(resp.read().decode(\"utf-8\"))\n            ice = data.get(\"iceServers\")\n            if isinstance(ice, list):\n                return ice\n            if isinstance(ice, dict):\n                return [ice]\n    except Exception as exc:\n        print(f\"[TURN] Warning: Could not generate direct TURN credentials: {exc}\", flush=True)\n    return []\n\n\ndef apply_turn_to_moonweb_config(turn_servers: list[dict], local_client: LocalMoonWebClient) -> bool:\n    if not turn_servers or not (MOONWEB_DIR / \"web-server\").exists():\n        return False\n    cfg_path = MOONWEB_DIR / \"server\" / \"config.json\"\n    cfg_path.parent.mkdir(parents=True, exist_ok=True)\n    cfg = None\n    if cfg_path.exists():\n        try:\n            cfg = json.loads(cfg_path.read_text(encoding=\"utf-8\"))\n        except Exception:\n            cfg = None\n    if not isinstance(cfg, dict):\n        try:\n            raw_cfg = subprocess.check_output(\n                [str(MOONWEB_DIR / \"web-server\"), \"print-config\"],\n                cwd=str(MOONWEB_DIR),\n                text=True,\n                timeout=8,\n            )\n            cfg = json.loads(raw_cfg)\n        except Exception as exc:\n            print(f\"[TURN] Could not inspect web-server config.json: {exc}\", flush=True)\n            return False\n\n    webrtc_cfg = cfg.setdefault(\"webrtc\", {})\n    existing = webrtc_cfg.get(\"ice_servers\")\n    if not isinstance(existing, list):\n        existing = []\n\n    # Remove old turn.cloudflare.com entries and prepend fresh ones\n    cleaned = [\n        s for s in existing\n        if not any(\"cloudflare.com\" in str(u) for u in (s.get(\"urls\") if isinstance(s.get(\"urls\"), list) else [s.get(\"urls\")]))\n    ]\n    webrtc_cfg[\"ice_servers\"] = list(turn_servers) + cleaned\n    cfg_path.write_text(json.dumps(cfg, indent=2), encoding=\"utf-8\")\n\n    # Restart local web-server cleanly so it reloads server/config.json with TURN UDP enabled\n    subprocess.run([\"pkill\", \"-f\", str(MOONWEB_DIR / \"web-server\")], check=False)\n    time.sleep(0.8)\n    subprocess.Popen(\n        [str(MOONWEB_DIR / \"web-server\"), \"--bind-address\", \"0.0.0.0:8081\"],\n        cwd=str(MOONWEB_DIR),\n        stdout=open(\"/tmp/webserver.log\", \"ab\"),\n        stderr=subprocess.STDOUT,\n    )\n    time.sleep(1.2)\n    local_client.cookie = \"\"\n    local_client.login()\n    print(\"[TURN] Injected Cloudflare TURN UDP servers into Moonlight-Web streamer!\", flush=True)\n    return True\n\n\ndef _ws_connect_kwargs(websockets_mod, headers: dict[str, str] | None = None) -> dict:\n    kwargs: dict = {\"max_size\": None, \"ping_interval\": 20, \"ping_timeout\": 20}\n    if headers:\n        sig = inspect.signature(websockets_mod.connect)\n        if \"additional_headers\" in sig.parameters:\n            kwargs[\"additional_headers\"] = headers\n        else:\n            kwargs[\"extra_headers\"] = headers\n    return kwargs\n\n\nasync def bridge_stream_ws(stream_id: str, local_client: LocalMoonWebClient) -> None:\n    import websockets\n\n    cookie = await asyncio.to_thread(local_client.ensure_cookie)\n    local_url = f\"{local_ws_origin()}/api/host/stream\"\n    remote_url = build_ws_url(\"colab_stream\", {\"streamId\": stream_id})\n\n    local_headers = {\"Cookie\": cookie} if cookie else None\n    try:\n        async with websockets.connect(local_url, **_ws_connect_kwargs(websockets, local_headers)) as ws_local, \\\n                   websockets.connect(remote_url, **_ws_connect_kwargs(websockets)) as ws_remote:\n\n            async def pump(src, dst) -> None:\n                try:\n                    async for frame in src:\n                        await dst.send(frame)\n                except Exception:\n                    pass\n                finally:\n                    try:\n                        await dst.close()\n                    except Exception:\n                        pass\n\n            await asyncio.gather(pump(ws_remote, ws_local), pump(ws_local, ws_remote))\n    except Exception as exc:\n        print(f\"[StreamBridge] Session {stream_id[:8]} ended: {exc}\", flush=True)\n\n\nasync def handle_http_rpc(ws, msg: dict, local_client: LocalMoonWebClient) -> None:\n    req_id = msg.get(\"reqId\", \"\")\n    method = str(msg.get(\"method\") or \"GET\").upper()\n    path = str(msg.get(\"path\") or \"/api\")\n    headers = msg.get(\"headers\") if isinstance(msg.get(\"headers\"), dict) else {}\n    body = base64.b64decode(msg.get(\"bodyB64\") or \"\") if msg.get(\"bodyB64\") else b\"\"\n\n    status, resp_headers, resp_body = await asyncio.to_thread(\n        local_client.request, method, path, headers, body\n    )\n    reply = {\n        \"type\": \"http_res\",\n        \"reqId\": req_id,\n        \"status\": status,\n        \"headers\": resp_headers,\n        \"bodyB64\": base64.b64encode(resp_body).decode(\"ascii\") if resp_body else \"\",\n    }\n    try:\n        await ws.send(json.dumps(reply))\n    except Exception:\n        pass\n\n\nasync def relay_main() -> None:\n    import websockets\n\n    if not (RELAY_URL and ROOM_ID and TOKEN):\n        print(\"[ERROR] COLABGG_RELAY_URL, COLABGG_ROOM_ID, and COLABGG_TOKEN are required.\", flush=True)\n        sys.exit(1)\n\n    local_client = LocalMoonWebClient()\n    await asyncio.to_thread(local_client.login)\n\n    direct_turn = await asyncio.to_thread(fetch_direct_cloudflare_turn)\n    if direct_turn:\n        local_client.turn_applied = await asyncio.to_thread(apply_turn_to_moonweb_config, direct_turn, local_client)\n\n    gpu_name, driver_ver, priv_ip = await asyncio.to_thread(query_gpu_telemetry)\n    backoff = 2\n\n    while True:\n        try:\n            ws_url = build_ws_url(\"colab\")\n            async with websockets.connect(ws_url, **_ws_connect_kwargs(websockets)) as ws:\n                backoff = 2\n                apps = await asyncio.to_thread(local_client.get_apps)\n                await ws.send(json.dumps({\n                    \"type\": \"hello\",\n                    \"serverName\": \"ColabGG Cloud Gaming\",\n                    \"gpuName\": gpu_name,\n                    \"driverVersion\": driver_ver,\n                    \"resolution\": \"1920x1080@60\",\n                    \"privateIp\": priv_ip,\n                    \"turnActive\": local_client.turn_applied,\n                    \"apps\": apps,\n                }))\n                print(f\"[ColabGG Relay] Connected to room '{ROOM_ID}' at {RELAY_URL}!\", flush=True)\n\n                async def heartbeat_loop() -> None:\n                    tick = 0\n                    while True:\n                        await asyncio.sleep(8.0)\n                        tick += 1\n                        cur_apps = await asyncio.to_thread(local_client.get_apps) if tick % 3 == 0 else apps\n                        await ws.send(json.dumps({\n                            \"type\": \"heartbeat\",\n                            \"gpuName\": gpu_name,\n                            \"driverVersion\": driver_ver,\n                            \"privateIp\": priv_ip,\n                            \"turnActive\": local_client.turn_applied,\n                            \"apps\": cur_apps,\n                        }))\n\n                hb_task = asyncio.create_task(heartbeat_loop())\n                try:\n                    async for raw in ws:\n                        if not isinstance(raw, str):\n                            continue\n                        msg = json.loads(raw)\n                        mtype = msg.get(\"type\")\n                        if mtype == \"welcome\":\n                            worker_turn = msg.get(\"iceServers\") or []\n                            if worker_turn and not local_client.turn_applied:\n                                applied = await asyncio.to_thread(apply_turn_to_moonweb_config, worker_turn, local_client)\n                                if applied:\n                                    local_client.turn_applied = True\n                                    await ws.send(json.dumps({\"type\": \"heartbeat\", \"turnActive\": True}))\n                        elif mtype == \"http_req\":\n                            asyncio.create_task(handle_http_rpc(ws, msg, local_client))\n                        elif mtype == \"open_stream_ws\" and msg.get(\"streamId\"):\n                            asyncio.create_task(bridge_stream_ws(str(msg[\"streamId\"]), local_client))\n                finally:\n                    hb_task.cancel()\n        except asyncio.CancelledError:\n            raise\n        except Exception as exc:\n            print(f\"[ColabGG Relay] Disconnected ({exc}), retrying in {backoff}s...\", flush=True)\n            await asyncio.sleep(backoff)\n            backoff = min(backoff * 2, 20)\n\n\nif __name__ == \"__main__\":\n    asyncio.run(relay_main())\n\nPYRELAYEOF\n        chmod +x /content/colab_relay_agent.py\n        pkill -f \"colab_relay_agent.py\" >/dev/null 2>&1 || true\n        log_info \"Starting ColabGG Relay Agent -> $COLABGG_RELAY_URL (Room: $COLABGG_ROOM_ID)...\"\n        COLABGG_RELAY_URL=\"$COLABGG_RELAY_URL\" COLABGG_ROOM_ID=\"$COLABGG_ROOM_ID\" COLABGG_TOKEN=\"$COLABGG_TOKEN\" \\\n        CF_TURN_KEY_ID=\"$CF_TURN_KEY_ID\" CF_TURN_API_TOKEN=\"$CF_TURN_API_TOKEN\" WEB_USER=\"$WEB_USER\" WEB_PASS=\"$WEB_PASS\" \\\n        nohup python3 /content/colab_relay_agent.py >/tmp/colabgg_relay.log 2>&1 &\n\n        for _ in $(seq 1 15); do\n            if grep -q \"Connected to room\" /tmp/colabgg_relay.log 2>/dev/null; then\n                break\n            fi\n            sleep 1\n        done\n        log_info \"ColabGG Relay Room '$COLABGG_ROOM_ID' is LIVE!\"\n    else\n        # Fallback Quick Tunnel only if Relay Room mode is not used\n        if [ ! -x /usr/local/bin/cloudflared ]; then\n            run_step \"Install cloudflared binary\" \\\n                \"sudo wget -q --retry-connrefused --tries=3 '$CLOUDFLARED_URL' -O /usr/local/bin/cloudflared && sudo chmod +x /usr/local/bin/cloudflared\" || exit 1\n        fi\n        pkill -f \"cloudflared tunnel --protocol http2\" >/dev/null 2>&1 || true\n        rm -f /tmp/cloudflared.log\n        nohup /usr/local/bin/cloudflared tunnel --protocol http2 --url http://127.0.0.1:8081 >/tmp/cloudflared.log 2>&1 &\n        local tunnel_url=\"\"\n        for _ in $(seq 1 30); do\n            tunnel_url=$(grep -oE 'https://[-a-z0-9]+\\.trycloudflare\\.com' /tmp/cloudflared.log 2>/dev/null | head -n 1 || true)\n            [ -n \"$tunnel_url\" ] && break\n            sleep 1\n        done\n        echo \"Quick Tunnel URL: ${tunnel_url:-unavailable} (Login: $WEB_USER / $WEB_PASS)\"\n    fi\n\n    if [ \"$KEEP_ALIVE_LOOP\" = \"true\" ]; then\n        steam_check_loop\n    fi\n}\n\ntrap backup_user SIGINT SIGTERM\n\nmain \"$@\"\n"
PAIR_CONTENT = "#!/usr/bin/env bash\nset -uo pipefail\n\n# Usage:\n#   ./moon-pair.sh --auto     # Automatically add 127.0.0.1 & pair Moonlight Web with Sunshine\n#   ./moon-pair.sh 1234       # Pair a specific 4-digit Moonlight PIN with Sunshine\n#   ./moon-pair.sh            # Prompt interactively for PIN (or press Enter for --auto)\n\nARG=\"${1:-}\"\n\nif command -v sunshine >/dev/null 2>&1 && id user >/dev/null 2>&1; then\n    su - user -c \"sunshine --creds '${SUNSHINE_USER:-admin}' '${SUNSHINE_PASS:-admin}'\" >/dev/null 2>&1 || true\nfi\n\nif [ -z \"$ARG\" ]; then\n    printf \"Enter Moonlight PIN (or press Enter for --auto): \"\n    read -r ARG || ARG=\"--auto\"\n    if [ -z \"$ARG\" ] || [ \"$ARG\" = \"auto\" ]; then\n        ARG=\"--auto\"\n    fi\nfi\n\npython3 - \"$ARG\" <<'PYEOF'\nimport base64\nimport json\nimport os\nimport ssl\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\n\nWEB_BASE = os.environ.get(\"MOONWEB_URL\", \"http://127.0.0.1:8081/api\")\nSUNSHINE_BASE = os.environ.get(\"SUNSHINE_URL\", \"https://127.0.0.1:47990/api\")\nWEB_USER = os.environ.get(\"WEB_USER\", \"admin\")\nWEB_PASS = os.environ.get(\"WEB_PASS\", \"admin\")\nSUN_USER = os.environ.get(\"SUNSHINE_USER\", \"admin\")\nSUN_PASS = os.environ.get(\"SUNSHINE_PASS\", \"admin\")\n\nssl_ctx = ssl.create_default_context()\nssl_ctx.check_hostname = False\nssl_ctx.verify_mode = ssl.CERT_NONE\n\n\ndef sunshine_req(path, payload):\n    auth = base64.b64encode(f\"{SUN_USER}:{SUN_PASS}\".encode()).decode()\n    data = json.dumps(payload).encode(\"utf-8\")\n    req = urllib.request.Request(\n        f\"{SUNSHINE_BASE}{path}\",\n        data=data,\n        method=\"POST\",\n        headers={\n            \"Content-Type\": \"application/json\",\n            \"Authorization\": f\"Basic {auth}\",\n        },\n    )\n    with urllib.request.urlopen(req, context=ssl_ctx, timeout=15) as resp:\n        return resp.status, resp.read().decode(\"utf-8\", errors=\"replace\")\n\n\ndef ensure_sunshine_ready():\n    for _ in range(25):\n        try:\n            req = urllib.request.Request(SUNSHINE_BASE.replace(\"/api\", \"/\"), method=\"GET\")\n            urllib.request.urlopen(req, context=ssl_ctx, timeout=3)\n            break\n        except urllib.error.HTTPError:\n            break\n        except Exception:\n            time.sleep(1)\n    try:\n        sunshine_req(\n            \"/password\",\n            {\n                \"currentUsername\": SUN_USER,\n                \"currentPassword\": SUN_PASS,\n                \"newUsername\": SUN_USER,\n                \"newPassword\": SUN_PASS,\n                \"confirmNewPassword\": SUN_PASS,\n            },\n        )\n    except Exception:\n        pass\n\n\ndef web_req(path, method=\"GET\", payload=None, cookie=\"\", timeout=15):\n    headers = {}\n    if cookie:\n        headers[\"Cookie\"] = cookie\n    data = None\n    if payload is not None:\n        headers[\"Content-Type\"] = \"application/json\"\n        data = json.dumps(payload).encode(\"utf-8\")\n    req = urllib.request.Request(f\"{WEB_BASE}{path}\", data=data, method=method, headers={**headers})\n    return urllib.request.urlopen(req, timeout=timeout)\n\n\ndef login_moonweb():\n    for _ in range(20):\n        try:\n            resp = web_req(\"/login\", method=\"POST\", payload={\"name\": WEB_USER, \"password\": WEB_PASS})\n            raw_cookies = resp.headers.get_all(\"Set-Cookie\") or []\n            cookie = \"; \".join(c.split(\";\", 1)[0] for c in raw_cookies if \"=\" in c)\n            return cookie\n        except urllib.error.HTTPError as e:\n            if e.code in (401, 403):\n                raise RuntimeError(f\"Moonlight Web login rejected for user '{WEB_USER}' (HTTP {e.code}).\")\n            time.sleep(1)\n        except Exception:\n            time.sleep(1)\n    raise RuntimeError(\"Moonlight Web server (:8081) did not become ready in time.\")\n\n\ndef main():\n    mode = sys.argv[1] if len(sys.argv) > 1 else \"--auto\"\n    ensure_sunshine_ready()\n\n    if mode != \"--auto\":\n        pin = mode.strip()\n        status, body = sunshine_req(\"/pin\", {\"pin\": pin, \"name\": \"moonlight\"})\n        print(f\"Sunshine PIN response ({status}): {body}\")\n        return 0\n\n    cookie = login_moonweb()\n\n    resp = web_req(\"/hosts\", method=\"GET\", cookie=cookie, timeout=15)\n    first_line = resp.readline().decode(\"utf-8\").strip()\n    resp.close()\n    hosts_obj = json.loads(first_line) if first_line else {\"hosts\": []}\n    hosts = hosts_obj.get(\"hosts\", [])\n\n    target_host = None\n    for h in hosts:\n        if h.get(\"address\") in (\"127.0.0.1\", \"localhost\") or len(hosts) == 1:\n            target_host = h\n            break\n\n    if target_host is None:\n        resp = web_req(\n            \"/host\",\n            method=\"POST\",\n            payload={\"address\": \"127.0.0.1\", \"http_port\": None},\n            cookie=cookie,\n            timeout=20,\n        )\n        target_host = json.loads(resp.read().decode(\"utf-8\"))[\"host\"]\n\n    host_id = target_host[\"host_id\"]\n\n    try:\n        web_req(\n            \"/host\",\n            method=\"PATCH\",\n            payload={\"host_id\": host_id, \"change_owner\": True, \"owner\": None},\n            cookie=cookie,\n            timeout=10,\n        )\n    except Exception:\n        pass\n\n    if target_host.get(\"paired\") == \"Paired\":\n        print(f\"[Auto-Pair] Host #{host_id} (127.0.0.1) is already Paired!\")\n        return 0\n\n    pair_resp = web_req(\n        \"/pair\",\n        method=\"POST\",\n        payload={\"host_id\": host_id},\n        cookie=cookie,\n        timeout=60,\n    )\n    stage1_raw = pair_resp.readline().decode(\"utf-8\").strip()\n    if not stage1_raw:\n        raise RuntimeError(\"Empty stage 1 response from /api/pair\")\n    stage1 = json.loads(stage1_raw)\n    if isinstance(stage1, str) or \"Pin\" not in stage1:\n        raise RuntimeError(f\"Failed to initiate pair (stage 1): {stage1}\")\n\n    pin = stage1[\"Pin\"]\n    print(f\"[Auto-Pair] Generated Moonlight PIN: {pin} -> Submitting to Sunshine...\")\n    sunshine_req(\"/pin\", {\"pin\": pin, \"name\": \"moonlight-web\"})\n\n    stage2_raw = pair_resp.readline().decode(\"utf-8\").strip()\n    pair_resp.close()\n    if not stage2_raw:\n        raise RuntimeError(\"Empty stage 2 response from /api/pair\")\n    stage2 = json.loads(stage2_raw)\n    if isinstance(stage2, str) or \"Paired\" not in stage2:\n        raise RuntimeError(f\"Failed to complete pair (stage 2): {stage2}\")\n\n    print(f\"[Auto-Pair] Successfully paired Moonlight Web host #{host_id} with Sunshine!\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\nPYEOF\n"

pathlib.Path("/content/colab-moonweb.sh").write_bytes(SCRIPT_CONTENT.encode("utf-8"))
pathlib.Path("/content/moon-pair.sh").write_bytes(PAIR_CONTENT.encode("utf-8"))
os.chmod("/content/colab-moonweb.sh", 0o755)
os.chmod("/content/moon-pair.sh", 0o755)

relay_url = Relay_Server_URL.strip().rstrip("/")
room_id = Custom_Room_ID.strip() or ("gg-" + secrets.token_urlsafe(8).replace("-", "").replace("_", "")[:10].lower())
room_token = secrets.token_urlsafe(24)

env = os.environ.copy()
env["COLABGG_RELAY_URL"] = relay_url
env["COLABGG_ROOM_ID"] = room_id
env["COLABGG_TOKEN"] = room_token
env["CF_TURN_KEY_ID"] = CF_TURN_KEY_ID.strip()
env["CF_TURN_API_TOKEN"] = CF_TURN_API_TOKEN.strip()
env["CF_TUNNEL_TOKEN"] = CF_ZeroTrust_Tunnel_Token.strip()
env["RESTORE_BACKUP"] = "true" if Restore_Backup_From_Drive else "false"
env["BACKUP_ON_EXIT"] = "true" if Backup_To_Drive_On_Stop else "false"
env["KEEP_ALIVE_LOOP"] = "false"

print("⏳ Đang cài đặt môi trường GPU, Sunshine, Moonlight-Web & kết nối Cloudflare Relay...")
rc = subprocess.call(["bash", "/content/colab-moonweb.sh"], env=env)
if rc != 0:
    raise RuntimeError(f"Setup exited with code {rc}. Check /tmp/colab_gaming_setup.log")

bundle_obj = {"mode": "relay", "serverUrl": relay_url, "roomId": room_id, "serverToken": room_token}
connect_bundle = "COLABGG_CONNECT: " + json.dumps(bundle_obj)
magic_b64 = base64.b64encode(json.dumps(bundle_obj).encode("utf-8")).decode("ascii")
magic_link = f"{relay_url}/#connect={magic_b64}"

display(Javascript(f"navigator.clipboard?.writeText({json.dumps(connect_bundle)}).catch(() => {{}});"))

card_html = f"""<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;background:#0f172a;border:1px solid #334155;border-radius:14px;padding:22px;max-width:760px;box-shadow:0 10px 25px -5px rgba(0,0,0,0.5);color:#f8fafc;box-sizing:border-box;margin-top:10px;">
  <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:12px;flex-wrap:wrap;gap:8px;">
    <div style="display:flex;align-items:center;gap:10px;">
      <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#10b981;box-shadow:0 0 10px #10b981;"></span>
      <h2 style="margin:0;font-size:18px;font-weight:700;color:#f8fafc;">ColabGG Cloud Gaming Server is Ready</h2>
    </div>
    <a href="{html.escape(magic_link)}" target="_blank" style="text-decoration:none;background:linear-gradient(135deg,#059669,#10b981);color:#022c22;font-weight:800;padding:7px 14px;border-radius:8px;font-size:12.5px;">🚀 Mở Web Client &amp; Kết nối ngay</a>
  </div>
  <p style="margin:0 0 16px;color:#94a3b8;font-size:13px;line-height:1.5;">Chuỗi <b>Auto-Connect Bundle</b> đã được copy sẵn vào Clipboard! Mở Web Client và bấm <b>Dán từ Clipboard</b> (hoặc bấm thẳng nút màu xanh ở góc phải trên).</p>
  <div style="margin-bottom:12px;">
    <label style="display:block;font-size:11px;font-weight:700;color:#cbd5e1;text-transform:uppercase;margin-bottom:5px;">Auto Connect Bundle</label>
    <div style="display:flex;gap:8px;">
      <input id="cgg-bundle" readonly value="{html.escape(connect_bundle)}" style="flex:1;padding:9px 12px;background:#020617;border:1px solid #334155;border-radius:8px;color:#38bdf8;font:12px ui-monospace,Consolas,monospace;outline:none;">
      <button onclick="navigator.clipboard.writeText(document.getElementById('cgg-bundle').value);this.textContent='Copied ✓'" style="padding:9px 14px;background:#1e293b;border:1px solid #475569;border-radius:8px;color:#f8fafc;font-size:12px;font-weight:600;cursor:pointer;">Copy Bundle</button>
    </div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;">
    <div>
      <label style="display:block;font-size:11px;font-weight:700;color:#cbd5e1;text-transform:uppercase;margin-bottom:5px;">Room ID</label>
      <input readonly value="{html.escape(room_id)}" style="width:100%;padding:8px 11px;background:#020617;border:1px solid #334155;border-radius:8px;color:#38bdf8;font:12px ui-monospace,Consolas,monospace;box-sizing:border-box;">
    </div>
    <div>
      <label style="display:block;font-size:11px;font-weight:700;color:#cbd5e1;text-transform:uppercase;margin-bottom:5px;">Connection Token</label>
      <input readonly value="{html.escape(room_token)}" style="width:100%;padding:8px 11px;background:#020617;border:1px solid #334155;border-radius:8px;color:#38bdf8;font:12px ui-monospace,Consolas,monospace;box-sizing:border-box;">
    </div>
  </div>
</div>"""
display(HTML(card_html))

# Activate VaultBox-style silent audio & auto-reconnect Keep-Alive engine
keepalive_ui = """<div style="margin-top:10px;max-width:760px;background:#0f172a;border:1px solid #1e293b;border-radius:10px;padding:11px 15px;color:#38bdf8;font-family:sans-serif;font-size:12.5px;display:flex;align-items:center;gap:10px;"><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:#38bdf8;box-shadow:0 0 8px #38bdf8;"></span><span><b>ColabGG Keep-Alive Engine Active:</b> Silent WebAudio oscillator &amp; auto-reconnect active. Nhấn nút Stop cell nếu muốn sao lưu lên Google Drive.</span></div><script>(() => { try { const ctx = new (window.AudioContext || window.webkitAudioContext)(); const osc = ctx.createOscillator(); const gain = ctx.createGain(); gain.gain.value = 0.00001; osc.connect(gain); gain.connect(ctx.destination); osc.start(); } catch(e) {} setInterval(() => { try { const btn = document.querySelector('colab-connect-button'); if (btn && btn.shadowRoot) { const inner = btn.shadowRoot.querySelector('#connect'); if (inner && inner.innerText.includes('Reconnect')) inner.click(); } } catch(e) {} }, 30000); })();</script>"""
display(HTML(keepalive_ui))

start_ts = time.time()
try:
    while True:
        time.sleep(30)
        # Ensure Steam, Sunshine, and Relay Agent stay running
        if subprocess.call(["pgrep", "-u", "user", "-x", "sunshine"], stdout=subprocess.DEVNULL) != 0:
            subprocess.Popen(["su", "-", "user", "-c", "DISPLAY=:1 nohup sunshine >/tmp/sunshine.log 2>&1 &"])
        if subprocess.call(["pgrep", "-f", "colab_relay_agent.py"], stdout=subprocess.DEVNULL) != 0:
            subprocess.Popen(["python3", "/content/colab_relay_agent.py"], env=env, stdout=open("/tmp/colabgg_relay.log", "ab"), stderr=subprocess.STDOUT)
        _ = sum(math.sin(i) for i in range(1000))
except KeyboardInterrupt:
    print("\n⏹️ Đã nhận lệnh dừng từ người dùng.")
    if Backup_To_Drive_On_Stop and os.path.isdir("/content/drive/MyDrive"):
        print("⏳ Đang sao lưu /home/user -> /content/drive/MyDrive/backup.tar.gz ...")
        subprocess.call("pkill -KILL -u user || true; tar -cf - -C /home/user ./ | pigz -6 > /content/drive/MyDrive/backup.tar.gz", shell=True)
        print("✅ Sao lưu hoàn tất!")


In [ ]:
#@title 💾 Sao lưu thủ công `/home/user` lên Google Drive (Tùy chọn) { display-mode: "form" }
import os
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
print('⏳ Đang nén /home/user -> /content/drive/MyDrive/backup.tar.gz ...')
!pkill -KILL -u user 2>/dev/null || true
!tar -cf - -C /home/user ./ | pigz -6 > /content/drive/MyDrive/backup.tar.gz
!ls -lh /content/drive/MyDrive/backup.tar.gz
print('✅ Sao lưu hoàn tất!')
